<a href="https://colab.research.google.com/github/Hayat188/Healthcare-Data-Analysis/blob/main/Healthcare_ETL_and_FHIR_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import hashlib

raw_data = {
    'Patient_ID': ['PT-101', 'PT-102', 'PT-103', 'PT-104', 'PT-105'],
    'Full_Name': ['Ahmed Hassan', 'Sara Ali', 'Mona Mahmoud', 'Omar Khaled', 'Noha Ibrahim'],
    'Systolic_BP': [118, 145, 132, 160, 115],
    'Diastolic_BP': [78, 92, 85, 100, 75],
    'ICD10_Code': ['E11.9', 'I10', 'E11.9', 'I10', 'Z00.00']
}

df_clinical = pd.DataFrame(raw_data)


def anonymize_name(name):
    return hashlib.sha256(name.encode()).hexdigest()[:10]

df_clinical['Patient_Hash'] = df_clinical['Full_Name'].apply(anonymize_name)
df_clinical.drop(columns=['Full_Name'], inplace=True)
def classify_bp(sys, dia):

    if sys < 120 and dia < 80:
        return 'Normal'
    elif 120 <= sys <= 129 and dia < 80:
        return 'Elevated'
    elif 130 <= sys <= 139 or 80 <= dia <= 89:
        return 'Hypertension Stage 1'
    else:
        return 'Hypertension Stage 2'

df_clinical['BP_Status'] = df_clinical.apply(lambda row: classify_bp(row['Systolic_BP'], row['Diastolic_BP']), axis=1)


print("(ETL Output)")
print(df_clinical)

(ETL Output)
  Patient_ID  Systolic_BP  Diastolic_BP ICD10_Code Patient_Hash  \
0     PT-101          118            78      E11.9   4167ac1018   
1     PT-102          145            92        I10   bf6b29b9a3   
2     PT-103          132            85      E11.9   4e99169622   
3     PT-104          160           100        I10   99b1e1ce62   
4     PT-105          115            75     Z00.00   21f5e98e65   

              BP_Status  
0                Normal  
1  Hypertension Stage 2  
2  Hypertension Stage 1  
3  Hypertension Stage 2  
4                Normal  


In [9]:
import json

fhir_resources = []

for index, row in df_clinical.iterrows():
    fhir_patient = {
        "resourceType": "Patient",
        "id": row['Patient_ID'],
        "identifier": [{
            "system": "http://hospital.org/hashing",
            "value": row['Patient_Hash']
        }],
        "medicalStatus": {
            "bloodPressure": f"{row['Systolic_BP']}/{row['Diastolic_BP']}",
            "statusCategory": row['BP_Status'],
            "icd10": row['ICD10_Code']
        }
    }
    fhir_resources.append(fhir_patient)

print(" FHIR JSON ")
print(json.dumps(fhir_resources[0], indent=4))

 FHIR JSON 
{
    "resourceType": "Patient",
    "id": "PT-101",
    "identifier": [
        {
            "system": "http://hospital.org/hashing",
            "value": "4167ac1018"
        }
    ],
    "medicalStatus": {
        "bloodPressure": "118/78",
        "statusCategory": "Normal",
        "icd10": "E11.9"
    }
}
